# Hisaab Aunty: Hinglish training

**Press Runtime > Run all.** Safe to re-run any time: if Colab disconnects, run it again and it carries on from the last saved step. Everything is saved in your Google Drive folder, not in Colab.

| Cell | What it does |
|---|---|
| 1 | Connects Drive, finds your model, sets the folders |
| 2 | Installs the tools and downloads our code and examples from GitHub |
| 3 | Prepares the training files **once** and saves them in Drive |
| 4 | Trains. Saves progress every 50 steps. Writes the finished model to Drive |
| 5 | Writes the data credits and gives the model an exam |

**Before you start:** Runtime > Change runtime type > T4 GPU. Your model folder (with config.json, tokenizer.json, tokenizer_config.json and model.safetensors) must be in Google Drive as My Drive/Sent_model.

Saved in Drive next to your model: Hisaab_work/mix (training files), Hisaab_work/checkpoints (progress), Sent_model_hinglish (the finished model).

In [ ]:
# 1. Setup: Drive, your model, and where everything is saved
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import os, json, random, re, shutil

BASE = next((p for p in ['/content/drive/MyDrive/Sent_model', '/content/drive/MyDrive/Hisaab/Sent_model'] if os.path.exists(p + '/model.safetensors')), None)
assert BASE, 'Model folder not found. Put it in Google Drive as My Drive/Sent_model'
PARENT = os.path.dirname(BASE)
MIX = PARENT + '/Hisaab_work/mix'              # training files (made once)
CKPT = PARENT + '/Hisaab_work/checkpoints'     # training progress (saved every 50 steps)
OUT = PARENT + '/Sent_model_hinglish'          # the finished model
EPOCHS = 2                                     # passes over the examples (1 = faster first try)

missing = [f for f in ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json'] if not os.path.exists(BASE + '/' + f)]
assert not missing, 'MISSING in ' + BASE + ': ' + ', '.join(missing) + '. Upload it from your PC folder Documents/Sent_model.'
print('model:', BASE, '\nsaves to:', PARENT)

In [ ]:
# 2. Install the tools and download our code and examples
!pip install -q -U transformers peft trl datasets accelerate
!pip uninstall -y -q torchao
!rm -rf /content/repo && git clone -q https://github.com/sohail807/hisaab_aunty /content/repo
!ls /content/repo/prototype/training/data

In [ ]:
# 3. Prepare the training files. Only the first time: they are saved in Drive and reused after that.
R = '/content/repo/prototype/training/data'

if os.path.exists(MIX + '/train.jsonl'):
    print('Training files already saved in Drive, reusing them:', MIX)
else:
    from datasets import load_dataset
    random.seed(7)
    read = lambda p: [json.loads(l) for l in open(p, encoding='utf-8')]
    train, val = read(R + '/train.jsonl'), read(R + '/val.jsonl')
    systems = [r['messages'][0]['content'] for r in train]          # the exact instruction the app sends

    # Public chat lines used as "do nothing" examples. Lines about money, banks, greetings or tax are left out.
    SKIP = re.compile(r"(\d|\u20b9|\brs\b|rupee|rupay|paisa|paise|kharch|bacha|budget|hisaab|bill|udhaar|salary|kamai|bank|loan|emi|khata|price|amount|balance|account|limit|spend|spent|expense|save|saving|goal|streak|recap|undo|delete|cancel|transfer|payment|paid|\bpay\b|card|cash|upi|refund|charge|fee|tax|invest|share|stock|crypto|fund|insurance|\bhi\b|hello|namaste|kaise|kaisi|thank|shukriya|bye|good (morning|night)|aunty|http|www|@)", re.I)
    def texts(x):
        if isinstance(x, str): yield x
        elif isinstance(x, dict):
            for v in x.values(): yield from texts(v)
        elif isinstance(x, (list, tuple)):
            for v in x: yield from texts(v)
    def clean(rows, n):
        out = set()
        for i, row in enumerate(rows):
            if i >= n: break
            for s in texts(row):
                for ln in s.split('\n'):
                    ln = re.sub(r'^\s*(person\s*\d*|user|speaker\s*\d*|human|assistant|[AB])\s*[:\-]\s*', '', ln, flags=re.I).strip()
                    if 3 <= len(ln.split()) <= 14 and re.fullmatch(r"[A-Za-z ,.'?!\-]+", ln) and not SKIP.search(ln): out.add(ln)
        return sorted(out)

    sources, hinglish, bank = [], [], []
    try:
        hinglish = clean(load_dataset('Abhishekcr448/Hinglish-Everyday-Conversations-1M', split='train', streaming=True), 30000)
        sources.append('Abhishekcr448/Hinglish-Everyday-Conversations-1M (license reported: MIT; synthetic, made with GPT-4o-mini)')
    except Exception as e: print('SKIPPED Hinglish chat:', repr(e)[:200])
    for name in ['PolyAI/banking77', 'legacy-datasets/banking77']:
        try:
            bank = clean(({'t': r['text']} for r in load_dataset(name, split='train')), 100000)
            sources.append(name + ' (Banking77, license reported: CC BY 4.0)'); break
        except Exception as e: print('could not load', name, repr(e)[:150])
    print('public lines found: Hinglish', len(hinglish), '| bank', len(bank))

    neg = lambda t: {'id': 'real_neg', 'intent': 'unknown', 'messages': [{'role': 'system', 'content': random.choice(systems)}, {'role': 'user', 'content': t}, {'role': 'assistant', 'content': '{"intent":"unknown"}'}]}
    random.shuffle(hinglish); random.shuffle(bank)
    held = hinglish[:40] + bank[:40]                                  # kept aside for the exam, never trained on
    hinglish, bank = hinglish[40:], bank[40:]
    mix_train = train + [neg(t) for t in hinglish[:1500]] + [neg(t) for t in bank[:1000]]
    mix_val = val + [neg(t) for t in hinglish[1500:1560]] + [neg(t) for t in bank[1000:1040]]
    random.shuffle(mix_train); random.shuffle(mix_val)
    os.makedirs(MIX, exist_ok=True)
    for name, rows in (('train', mix_train), ('val', mix_val), ('test_real', [neg(t) for t in held])):
        open(MIX + '/' + name + '.jsonl', 'w', encoding='utf-8').write('\n'.join(json.dumps(r, ensure_ascii=False) for r in rows) + '\n')
    json.dump(sources, open(MIX + '/sources.json', 'w'))
    print('saved to Drive: train', len(mix_train), '| val', len(mix_val), '| exam lines', len(held))

## 4. Train
Saves progress to Drive every 50 steps. If Colab disconnects, press Run all again: cells 1 to 3 are quick and this cell continues from the last saved step. Keep this tab open and the laptop awake. A free T4 needs a few hours (not measured).

In [ ]:
# 4. Train. Continues from saved progress; writes the finished model straight to Drive.
!python /content/repo/prototype/training/train_lora.py --base "$BASE" --data "$MIX" --out "$OUT" --ckpt "$CKPT" --epochs $EPOCHS

In [ ]:
# 5. Credits file and an exam on messages the model never saw
sources = json.load(open(MIX + '/sources.json')) if os.path.exists(MIX + '/sources.json') else []
open(OUT + '/DATA_SOURCES.md', 'w').write('# Data used to train this model\n\n- Our own generated Hinglish intent data: https://github.com/sohail807/hisaab_aunty (prototype/training/gen_dataset.js)\n' + ''.join('- ' + s + '\n' for s in sources) + '\nCheck each dataset license before commercial use.\n')

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(OUT)
model = AutoModelForCausalLM.from_pretrained(OUT, dtype=torch.float16, device_map='auto')
test = [json.loads(l) for l in open(R + '/test.jsonl')]
step = max(1, len(test) // 60)
sample = [r for i, r in enumerate(test) if i % step == 0][:60] + [json.loads(l) for l in open(MIX + '/test_real.jsonl')]
def ask(r):
    prompt = tok.apply_chat_template(r['messages'][:2], tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**ids, max_new_tokens=120, do_sample=False)
    return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
def intent_of(s):
    m = re.search(r'\{.*\}', s, re.S)
    try: return json.loads(m.group(0)).get('intent') if m else None
    except Exception: return None
ok = valid = wrong_saves = n = real_n = real_ok = 0; bad = []
for r in sample:
    got = ask(r); gold = json.loads(r['messages'][2]['content'])['intent']; pred = intent_of(got); n += 1
    valid += pred is not None; ok += pred == gold
    if pred == 'log_spend' and gold != 'log_spend': wrong_saves += 1
    if r.get('id') == 'real_neg': real_n += 1; real_ok += pred == 'unknown'
    if pred != gold and len(bad) < 12: bad.append((r['messages'][1]['content'], gold, pred))
print(f'messages {n} | valid JSON {valid}/{n} | right intent {ok}/{n} ({100*ok//n}%)   [before training: about 50%]')
print(f'WRONG SAVES (said log_spend when it should not): {wrong_saves}   [must be 0]')
print(f'real unrelated chat correctly left alone: {real_ok}/{real_n}')
for b in bad: print('MISS', b)

## Done
The finished model is in Drive: Sent_model_hinglish (about 6 GB). Next on your PC: download it, convert it to GGUF, add it to models.json, and run: node training/eval_model.js --model <id> --n 70. Keep it only if it beats the old score and wrong saves stay at 0.